# 05 ETF 申赎套利回测（Backtrader）

标的：`511130.SH`  
区间：`2026-01-01 ~ 2026-02-28`

本 Notebook 基于清洗并合并后的 ETF/债券数据，构建一级市场申赎套利回测：
- 用 `pd.merge_asof` 对齐 ETF 与债券高频时间戳
- 基于 PCF 成分券数量 + 现金差额构建 IOPV
- 引入简化的应计利息模型（clean -> dirty）
- 用 `Backtrader` 策略类记录套利信号与现金流
- 输出交易日志、业绩指标、可视化图表


In [ ]:
from __future__ import annotations
from pathlib import Path
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import backtrader as bt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('etf_arb')


In [ ]:
# =========================
# 1) 配置
# =========================
ETF_PATH = Path('merged_outputs/511130_20260101_20260228_merged.csv')
BOND_PATHS = {
    '019742': Path('merged_outputs/019742_20260101_20260312_merged.csv'),
    '019776': Path('merged_outputs/019776_20260101_20260312_merged.csv'),
    '019789': Path('merged_outputs/019789_20260101_20260312_merged.csv'),
}
PCF_PATH = Path('测试题/511130_PCF清单_2024-12-31_2026-03-20.csv')
CASH_PATH = Path('测试题/511130_现金差额_2024-12-31_2026-03-20.csv')

TIME_COL = 'trade_time'
ETF_COST = 0.00005
BOND_COST = 0.000005
START_DATE = pd.Timestamp('2026-01-01')
END_DATE = pd.Timestamp('2026-02-28 23:59:59')
ALIGN_TOL = pd.Timedelta('1s')

ACCRUED_INTEREST_BP_PER_DAY = 0.8  # 简化模型：每日应计 0.8bp（按净价百分位）


In [ ]:
# =========================
# 2) 读取数据与基础检查
# =========================
def robust_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    return pd.read_csv(path)

etf_df = robust_read_csv(ETF_PATH)
bond_dfs = {k: robust_read_csv(v) for k, v in BOND_PATHS.items()}
pcf_df = robust_read_csv(PCF_PATH)
cash_df = robust_read_csv(CASH_PATH)

for name, df in [('ETF', etf_df), ('PCF', pcf_df), ('CASH', cash_df)] + [(f'BOND_{k}', v) for k,v in bond_dfs.items()]:
    logger.info('%s shape=%s', name, df.shape)

# 时间字段处理
etf_df[TIME_COL] = pd.to_datetime(etf_df[TIME_COL], errors='coerce')
etf_df = etf_df.dropna(subset=[TIME_COL]).sort_values(TIME_COL)
etf_df = etf_df[(etf_df[TIME_COL] >= START_DATE) & (etf_df[TIME_COL] <= END_DATE)].copy()

for k, bdf in bond_dfs.items():
    bdf[TIME_COL] = pd.to_datetime(bdf[TIME_COL], errors='coerce')
    bdf = bdf.dropna(subset=[TIME_COL]).sort_values(TIME_COL)
    bdf = bdf[(bdf[TIME_COL] >= START_DATE) & (bdf[TIME_COL] <= END_DATE)].copy()
    bond_dfs[k] = bdf

logger.info('ETF range: %s -> %s', etf_df[TIME_COL].min(), etf_df[TIME_COL].max())
for k, bdf in bond_dfs.items():
    logger.info('Bond %s range: %s -> %s', k, bdf[TIME_COL].min(), bdf[TIME_COL].max())


In [ ]:
# =========================
# 3) 构建 PCF 篮子权重 + 现金差额（日频）
# =========================
pcf_df['date'] = pd.to_datetime(pcf_df['date'], errors='coerce')
pcf_df['stock_code'] = pcf_df['stock_code'].astype(str).str.extract(r'(\d+)')[0].str.zfill(6)

target_bonds = ['019742', '019776', '019789']
pcf_df = pcf_df[pcf_df['stock_code'].isin(target_bonds)].copy()
pcf_df['stock_amount'] = pd.to_numeric(pcf_df['stock_amount'], errors='coerce').fillna(0.0)

pcf_daily = pcf_df.groupby(['date','stock_code'], as_index=False)['stock_amount'].sum()

cash_df['pre_date'] = pd.to_datetime(cash_df['pre_date'], errors='coerce')
cash_df['cash_component'] = pd.to_numeric(cash_df['cash_component'], errors='coerce').fillna(0.0)
cash_daily = cash_df[['pre_date','cash_component']].dropna().rename(columns={'pre_date':'date'})

display(pcf_daily.head())
display(cash_daily.head())


In [ ]:
# =========================
# 4) 高频时间对齐（merge_asof）+ IOPV构建
# =========================
def pick_price_col(df: pd.DataFrame) -> str:
    candidates = ['last','close','bid_price1','ask_price1']
    for c in candidates:
        if c in df.columns:
            return c
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        raise ValueError('No numeric price-like column found')
    return num_cols[0]

etf_price_col = pick_price_col(etf_df)
bond_price_cols = {k: pick_price_col(v) for k,v in bond_dfs.items()}
logger.info('ETF price col=%s; bond cols=%s', etf_price_col, bond_price_cols)

aligned = etf_df[[TIME_COL, etf_price_col]].rename(columns={etf_price_col:'etf_close'}).copy()
aligned = aligned.sort_values(TIME_COL)

for code, bdf in bond_dfs.items():
    col = bond_price_cols[code]
    tmp = bdf[[TIME_COL, col]].rename(columns={col:f'{code}_clean_price'}).sort_values(TIME_COL)
    aligned = pd.merge_asof(
        aligned.sort_values(TIME_COL),
        tmp.sort_values(TIME_COL),
        on=TIME_COL,
        direction='nearest',
        tolerance=ALIGN_TOL
    )

# 逐时点挂接日频 PCF & cash_component
aligned['date'] = aligned[TIME_COL].dt.floor('D')

weight_wide = pcf_daily.pivot(index='date', columns='stock_code', values='stock_amount').reset_index()
weight_wide.columns = ['date'] + [f'{c}_amount' for c in weight_wide.columns[1:]]

aligned = aligned.merge(weight_wide, on='date', how='left')
aligned = aligned.merge(cash_daily, on='date', how='left')
aligned['cash_component'] = aligned['cash_component'].fillna(0.0)

# 简化应计利息模型：dirty = clean * (1 + bp_per_day*days/10000)
# days 采用相对回测起始日天数
aligned['days_from_start'] = (aligned['date'] - aligned['date'].min()).dt.days.clip(lower=0)
for code in target_bonds:
    cp = f'{code}_clean_price'
    dp = f'{code}_dirty_price'
    if cp in aligned.columns:
        aligned[cp] = pd.to_numeric(aligned[cp], errors='coerce')
        aligned[dp] = aligned[cp] * (1 + ACCRUED_INTEREST_BP_PER_DAY * aligned['days_from_start'] / 10000.0)
        aligned[f'{code}_amount'] = pd.to_numeric(aligned.get(f'{code}_amount', 0.0), errors='coerce').fillna(0.0)
    else:
        aligned[dp] = np.nan
        aligned[f'{code}_amount'] = 0.0

aligned['basket_value'] = sum(aligned.get(f'{c}_dirty_price', 0.0) * aligned.get(f'{c}_amount', 0.0) for c in target_bonds)
aligned['iopv_theory'] = aligned['basket_value'] + aligned['cash_component']
aligned['spread'] = aligned['etf_close'] - aligned['iopv_theory']

logger.info('Aligned shape=%s', aligned.shape)
print('merge后缺失值（前20列）:')
print(aligned.isna().mean().head(20))
display(aligned.head())


In [ ]:
# =========================
# 5) Backtrader 数据准备
# =========================
bt_df = aligned[[TIME_COL, 'etf_close', 'iopv_theory', 'spread', 'basket_value', 'cash_component']].copy()
bt_df = bt_df.dropna(subset=['etf_close', 'iopv_theory']).sort_values(TIME_COL)
bt_df = bt_df.drop_duplicates(subset=[TIME_COL], keep='last')

# Backtrader 标准 OHLCV 字段（这里用 etf_close 构造简化bar）
bt_feed_df = bt_df.rename(columns={TIME_COL:'datetime'}).copy()
for c in ['open','high','low','close']:
    bt_feed_df[c] = bt_feed_df['etf_close']
bt_feed_df['volume'] = 0.0
bt_feed_df['openinterest'] = 0.0
bt_feed_df = bt_feed_df.set_index('datetime')

print('bt_feed_df shape:', bt_feed_df.shape)
print('time range:', bt_feed_df.index.min(), '->', bt_feed_df.index.max())


In [ ]:
# =========================
# 6) Backtrader 策略
# =========================
class PandasArbData(bt.feeds.PandasData):
    lines = ('iopv_theory', 'spread', 'basket_value', 'cash_component', 'etf_close')
    params = (
        ('datetime', None),
        ('open', 'open'), ('high', 'high'), ('low', 'low'), ('close', 'close'),
        ('volume', 'volume'), ('openinterest', 'openinterest'),
        ('iopv_theory', 'iopv_theory'),
        ('spread', 'spread'),
        ('basket_value', 'basket_value'),
        ('cash_component', 'cash_component'),
        ('etf_close', 'etf_close'),
    )

class ETFArbitrageStrategy(bt.Strategy):
    params = dict(threshold=0.001, max_position=1_000_000, max_trade_size=10_000, initial_cash=10_000_000)

    def __init__(self):
        self.trade_logs = []
        self.cum_pnl = 0.0

    def next(self):
        dt = self.datas[0].datetime.datetime(0)
        etf_close = float(self.datas[0].etf_close[0])
        iopv = float(self.datas[0].iopv_theory[0])
        spread = etf_close - iopv
        basket_value = float(self.datas[0].basket_value[0])
        cash_component = float(self.datas[0].cash_component[0])

        # 交易规模（简化）：按 ETF 价格推导名义
        trade_size = min(self.p.max_trade_size, max(1, int(self.p.max_position / max(etf_close, 1e-8))))
        etf_value = trade_size * etf_close
        basket_notional = trade_size * iopv

        # 成本：ETF双边 + 债券双边
        tx_cost = etf_value * 2 * ETF_COST + basket_notional * 2 * BOND_COST

        pnl = 0.0
        side = 'NONE'

        # 溢价套利：卖ETF、买篮子、申购
        if spread - tx_cost / max(trade_size,1) > self.p.threshold:
            pnl = etf_value - basket_notional - tx_cost + cash_component
            side = 'PREMIUM_ARB'

        # 折价套利：买ETF、赎回卖篮子
        elif (-spread) - tx_cost / max(trade_size,1) > self.p.threshold:
            pnl = basket_notional - etf_value - tx_cost + cash_component
            side = 'DISCOUNT_ARB'

        if side != 'NONE':
            self.cum_pnl += pnl
            self.trade_logs.append({
                'trade_time': dt,
                'side': side,
                'etf_price': etf_close,
                'iopv': iopv,
                'spread': spread,
                'transaction_cost': tx_cost,
                'cash_component': cash_component,
                'pnl': pnl,
                'cumulative_pnl': self.cum_pnl,
                'trade_size': trade_size,
                'basket_value': basket_value,
            })


In [ ]:
# =========================
# 7) 运行回测 + 指标输出
# =========================
cerebro = bt.Cerebro(stdstats=False)
data_feed = PandasArbData(dataname=bt_feed_df)
cerebro.adddata(data_feed)

cerebro.addstrategy(ETFArbitrageStrategy, threshold=0.001, max_position=1_000_000, max_trade_size=10_000, initial_cash=10_000_000)
cerebro.broker.setcash(10_000_000)

cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', timeframe=bt.TimeFrame.Days)
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')

results = cerebro.run()
strat = results[0]

trade_log = pd.DataFrame(strat.trade_logs)
trade_log.to_csv('trade_log.csv', index=False)

print('Trade Count:', len(trade_log))
print('Sharpe:', strat.analyzers.sharpe.get_analysis())
print('DrawDown:', strat.analyzers.drawdown.get_analysis())
print('Returns:', strat.analyzers.returns.get_analysis())
print('TradeAnalyzer:', strat.analyzers.trades.get_analysis())

if len(trade_log) > 0:
    trade_log['trade_date'] = pd.to_datetime(trade_log['trade_time']).dt.date
    daily_trade_count = trade_log.groupby('trade_date').size().rename('trade_count')
    print('每日交易次数:')
    display(daily_trade_count.head(20))

display(trade_log.head())


In [ ]:
# =========================
# 8) 可视化
# =========================
plot_df = bt_df.copy().set_index(TIME_COL)

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

axes[0].plot(plot_df.index, plot_df['etf_close'], label='ETF Price')
axes[0].plot(plot_df.index, plot_df['iopv_theory'], label='IOPV Theory', alpha=0.8)
axes[0].set_title('ETF价格 vs IOPV')
axes[0].legend()

axes[1].plot(plot_df.index, plot_df['spread'], color='purple')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Spread 时间序列')

if len(trade_log) > 0:
    tl = trade_log.copy()
    tl['trade_time'] = pd.to_datetime(tl['trade_time'])
    tl = tl.sort_values('trade_time')
    axes[2].plot(tl['trade_time'], tl['cumulative_pnl'], color='green')
    axes[2].set_title('累计PnL')

    running_max = tl['cumulative_pnl'].cummax()
    dd = tl['cumulative_pnl'] - running_max
    axes[3].plot(tl['trade_time'], dd, color='red')
    axes[3].set_title('Drawdown')
else:
    axes[2].set_title('累计PnL（无交易）')
    axes[3].set_title('Drawdown（无交易）')

plt.tight_layout()
plt.show()
